In [ ]:
import asyncio
import json
import os
import time
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langsmith import Client
from pydantic import BaseModel, Field

from chat_app_backend_rag import _get_vectorstore, _load_and_split, add_documents_to_store, clear_collection, generate_output

PROJECT_ROOT = Path(r'D:\AI-ML\AI Projects\fastapi-chat-app')
load_dotenv(dotenv_path=PROJECT_ROOT / '.env')
nest_asyncio.apply()
os.environ['LANGSMITH_TRACING'] = 'true'

EVAL_DIR = PROJECT_ROOT / 'eval_files'
PDF_PATHS = sorted(EVAL_DIR.glob('*.pdf'))
EVAL_COLLECTION = 'langsmith_document_eval'
DATASET_NAME = 'personal-ai-assistant-document-rag-v1'

if len(PDF_PATHS) < 2:
    raise FileNotFoundError(f'Expected at least two PDFs in {EVAL_DIR}, found: {PDF_PATHS}')

client = Client()
llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0.0, max_tokens=512)
print('Evaluation documents:')
for path in PDF_PATHS:
    print(f'- {path.name}')
print('LangSmith key loaded:', bool(os.getenv('LANGSMITH_API_KEY')))



Evaluation documents:
- Bangladesh_Company_Law_Guide_BD_Chapter.pdf
- BUssiness information for RAG.pdf
LangSmith key loaded: True


In [3]:
all_chunks = []
for pdf_path in PDF_PATHS:
    chunks = _load_and_split([str(pdf_path)])
    for chunk in chunks:
        chunk.metadata['source'] = pdf_path.name
    all_chunks.extend(chunks)

print(f'Loaded {len(all_chunks)} chunks from {len(PDF_PATHS)} PDFs')
for source in sorted({chunk.metadata.get('source') for chunk in all_chunks}):
    count = sum(chunk.metadata.get('source') == source for chunk in all_chunks)
    print(f'- {source}: {count} chunks')

try:
    clear_collection(EVAL_COLLECTION)
except Exception:
    pass

add_documents_to_store([str(path) for path in PDF_PATHS], collection_name=EVAL_COLLECTION)
print(f'Indexed documents into Chroma collection: {EVAL_COLLECTION}')

Loaded 71 chunks from 2 PDFs
- BUssiness information for RAG.pdf: 26 chunks
- Bangladesh_Company_Law_Guide_BD_Chapter.pdf: 45 chunks
Indexed documents into Chroma collection: langsmith_document_eval


In [4]:
class GeneratedQA(BaseModel):
    question: str = Field(description='A specific question answerable from the supplied passage')
    answer: str = Field(description='A concise answer supported only by the supplied passage')
    evidence: str = Field(description='A short exact quote or faithful excerpt supporting the answer')


def make_candidate_examples(chunks_per_document: int = 5) -> list[dict]:
    candidates = []
    for source in sorted({chunk.metadata.get('source') for chunk in all_chunks}):
        source_chunks = [chunk for chunk in all_chunks if chunk.metadata.get('source') == source]
        selected = source_chunks[:chunks_per_document]
        for chunk in selected:
            prompt = PromptTemplate.from_template(
                '''
                Create one evaluation question and answer from this document passage.
                The question must require information in the passage, not outside knowledge.
                The answer must be concise, accurate, and contain no unsupported claims.
                Include a short evidence excerpt from the passage.

                Source: {source}
                Passage:
                {passage}
                '''
            ).format(source=source, passage=chunk.page_content)
            qa = llm.with_structured_output(GeneratedQA).invoke(prompt)
            candidates.append({
                'question': qa.question,
                'answer': qa.answer,
                'evidence': qa.evidence,
                'source': source,
                'page': chunk.metadata.get('page'),
            })
    return candidates[:10]

candidate_examples = make_candidate_examples()
print(f'Generated {len(candidate_examples)} candidate examples')
for idx, example in enumerate(candidate_examples, 1):
    print(f'[{idx}] Q: {example["question"]}')
    print(f'    A: {example["answer"]}')
    print(f'    SOURCE: {example["source"]}')
    print('---')


Generated 10 candidate examples
[1] Q: Who prepared the company profile document?
    A: Md Mahbubur Rahman
    SOURCE: BUssiness information for RAG.pdf
---
[2] Q: What is the title of section 7 in the document?
    A: AREAS OF EXPERTISE
    SOURCE: BUssiness information for RAG.pdf
---
[3] Q: When did PPrriimmee IITT begin its business operation?
    A: January 2003
    SOURCE: BUssiness information for RAG.pdf
---
[4] Q: Who formed the company and what are their qualifications?
    A: The company was formed by a group of professionals with vivid experience and wide exposure in Information Technology, including young qualified business graduates and qualified engineers from renowned universities worldwide.
    SOURCE: BUssiness information for RAG.pdf
---
[5] Q: What are the major building blocks of the company's long-term business partnership with its clients?
    A: Interpersonal relationship, reliability, assured quality, and target‑oriented modern technology.
    SOURCE: BUssines

In [5]:
reviewed_examples = candidate_examples

assert len(reviewed_examples) == 10, f'Expected 10 reviewed examples, got {len(reviewed_examples)}'
assert all(example['question'].strip() for example in reviewed_examples)
assert all(example['answer'].strip() for example in reviewed_examples)
assert len({example['source'] for example in reviewed_examples}) == len(PDF_PATHS)
print('Review checkpoint passed. Examples are ready for LangSmith.')

Review checkpoint passed. Examples are ready for LangSmith.


In [8]:
from concurrent.futures import ThreadPoolExecutor

answer_prompt = PromptTemplate.from_template(
    '''
    Answer the question using only the retrieved document context below.
    If the context does not answer the question, say that the documents do not contain enough information.
    Do not invent facts or citations. Keep the answer concise.

    Question: {question}
    Retrieved context, ordered from highest-ranked to lowest-ranked:
    {context}
    '''
)


def evaluate_target(inputs: dict) -> dict:
    question = inputs['question']
    started = time.perf_counter()

    def run_query():
        return asyncio.run(generate_output(question, _get_vectorstore(EVAL_COLLECTION)))

    with ThreadPoolExecutor(max_workers=1) as executor:
        context = executor.submit(run_query).result()

    response = llm.invoke(answer_prompt.format(question=question, context=context)).content
    return {
        'response': response,
        'context': context,
        'latency_seconds': round(time.perf_counter() - started, 3),
    }

sample_output = evaluate_target({'question': reviewed_examples[0]['question']})
assert sample_output['response']
assert sample_output['context']
print(json.dumps(sample_output, indent=2, ensure_ascii=False))


['COMPANY PROFILE \n  \n  \n  \n  \n  \n  \n  \n  \n  \n  \n  \n  \n  \n  \n  \nDOCUMENT  \n  \n  \nCompany profile  \n  \n  \n  \nPREPARED BY \n \n \nMd Mahbubur Rahman \n \n \n \nDATE \n  \n  \nNovember, 2017  \n  \n  \n  \n  \n  \n  \n \nCORPORATE OFFICE \n115522//22--MM  GGrreeeenn  RRooaadd  \nPPaanntthhaappaatthh  ,,  DDhhaakkaa--11220055  \nBBaannggllaaddeesshh  \nTTeelleepphhoonnee  ::  ++8888  0022  5588115522999900  \n  \n \nCORRESPONDENCE OFFICE \n115522//22--MM  GGrreeeenn  RRooaadd  \nPPaanntthhaappaatthh  ,,  DDhhaakkaa--11220055  \nBBaannggllaaddeesshh  \nTTeelleepphhoonnee  ::  ++8888  0022  5588115522999900', 'COMPANY PROFILE  \n   \n----------------------------------------------------------------------------------------------------------------------------------------------------------------------------- \nConfidential © 2014 APSIS All Rights Reserved Page 3 of 14 \n \n2. YOUR AUTOMATION PARTNER \n \n \nThe company has been formed by a group of professionals having viv

In [9]:
existing_datasets = list(client.list_datasets(dataset_name=DATASET_NAME))
if existing_datasets:
    dataset = existing_datasets[0]
    print(f'Reusing LangSmith dataset: {dataset.id}')
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description='Reviewed document-grounded QA examples for the Personal AI Assistant RAG pipeline.'
    )
    client.create_examples(
        inputs=[{'question': example['question']} for example in reviewed_examples],
        outputs=[
            {
                'answer': example['answer'],
                'evidence': example['evidence'],
                'source': example['source'],
                'page': example['page'],
            }
            for example in reviewed_examples
        ],
        dataset_id=dataset.id,
    )
    print(f'Created LangSmith dataset: {dataset.id}')

print(f'Dataset name: {DATASET_NAME}')

Created LangSmith dataset: bf9adc5a-e8ff-4beb-aa9d-a4802ecba6ca
Dataset name: personal-ai-assistant-document-rag-v1


In [10]:
class JudgeScore(BaseModel):
    score: float = Field(ge=0.0, le=1.0)
    reason: str


def make_judge(key: str, instructions: str):
    judge_prompt = PromptTemplate.from_template(
        '''
        You are evaluating a document-grounded AI assistant.
        Return a score from 0.0 to 1.0 and a short reason.

        Evaluation rule:
        {instructions}

        Question: {question}
        Reference answer: {reference_answer}
        Assistant answer: {answer}
        Retrieved context: {context}
        '''
    )

    def evaluator(inputs, outputs, reference_outputs):
        result = llm.with_structured_output(JudgeScore).invoke(
            judge_prompt.format(
                instructions=instructions,
                question=inputs.get('question', ''),
                reference_answer=(reference_outputs or {}).get('answer', ''),
                answer=outputs.get('response', ''),
                context=outputs.get('context', ''),
            )
        )
        return {'key': key, 'score': result.score, 'comment': result.reason}

    evaluator.__name__ = key
    return evaluator


correctness = make_judge(
    'answer_correctness',
    'Score how accurately the assistant answer matches the reference answer. Give 1 only when the answer is materially correct; penalize wrong or missing facts.',
)
answer_relevance = make_judge(
    'answer_relevance',
    'Score how directly and usefully the assistant answer addresses the question. Penalize unrelated, evasive, or excessively vague answers.',
)
faithfulness = make_judge(
    'faithfulness',
    'Score whether every important claim in the assistant answer is supported by the retrieved context. Penalize unsupported claims and contradictions.',
)
completeness = make_judge(
    'completeness',
    'Score whether the assistant answer covers the important parts of the reference answer without omitting material details.',
)
print('Evaluators ready: correctness, answer_relevance, faithfulness, completeness')

Evaluators ready: correctness, answer_relevance, faithfulness, completeness


In [11]:
evaluation = client.evaluate(
    evaluate_target,
    data=DATASET_NAME,
    evaluators=[correctness, answer_relevance, faithfulness, completeness],
    experiment_prefix='document-rag-v1',
    max_concurrency=1,
)

print('LangSmith evaluation started.')
print(evaluation)

View the evaluation results for experiment: 'document-rag-v1-dd47994c' at:
https://smith.langchain.com/o/8b96c356-c6bf-4cbd-9d1b-e231816f9207/datasets/bf9adc5a-e8ff-4beb-aa9d-a4802ecba6ca/compare?selectedSessions=3d45476a-1878-4de4-9784-b592101edc60




0it [00:00, ?it/s]

['COMPANY PROFILE  \n   \n----------------------------------------------------------------------------------------------------------------------------------------------------------------------------- \nConfidential © 2014 APSIS All Rights Reserved Page 2 of 14 \n \n1. INTRODUCTION \n \n \nPPrriimmee  IITT  provides automated solution for your business and industry. Depending on the size and \nfield of your organization, we have different products and services to meet your requirements. \nWe provide the optimum and customized solutions made for your organization.  \n \nPPrriimmee  IITT  began its business operation as a hardware and network solutions providing company \nin January 2003. \n \nPPrriimmee  IITT  is focusing exclusively in high quality and cost-effective software development and \nimplementation of services. We are advancing on a tremendous pace and with involvement of \nskilled and experienced people working in the organization. APSIS is currently doing business in \nGover

Error running evaluator <DynamicRunEvaluator answer_relevance> on run 01a0ba34-8a91-7db0-8bb4-b58fa5f9d5c9: BadRequestError("Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': 'Score: 1.0  \\nReason: The answer directly provides the requested date (“January\\u202f2003”), matches the reference answer, and includes a citation to the source, making it fully accurate and useful.'}}")
Traceback (most recent call last):
  File "c:\Users\Tanvir\AppData\Local\Programs\Python\Python313\Lib\site-packages\langsmith\evaluation\_runner.py", line 1672, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
        run=run,
        example=example,
        evaluator_run_id=evaluator_run_id,
    )
  File "c:\Users\Tanvir\AppData\Local\Programs\Python\Python313\Lib\site-packages\langsmith\evaluation\evaluator.py", line 370, in evaluate_ru

['Bangladesh Bank (‘BB’), the central  bank  of Bangladesh,  which regulations are compiled by \nthe BB in the Guidelines  for Foreign Exchange  Transactions Volume 1 & Volume 2 (2009), and \nupdated  by BB’s circulars  issued  from time to time (collectively,  the ‘FX Guidelines’); and \n(b) the Companies Act, 1994 of Bangladesh (Act No. XVIII of 1994) (‘CA 1994’). \n \nThe key Bangladeshi government agencies or regulator y bodies that impact or regulate foreig n \ncompanies and investors are:  \n(a) the Bangladesh Investment Development Authority (‘BIDA’), formerly known as the Board of \nInvestment, which facilitates foreign investment by advising foreign investors and assisting them \nwith utilities, land acquisition,  etc; \n(b) BB, Bangladesh’s central bank, which regu- lates the outward repatriation of capital and capital  \ngains; and \n(c) the Registrar of Joint Stock Companies and Firms (‘RJSC’), which registers both foreign \ncompanies establishing a place of business in Ban

...

["building a long-term business partnership with its clients where interpersonal relationship, \nreliability, assured quality and target oriented modern technology are the major building blocks.  \n \nIt is a company where professionals from both technical and functional field group together with \nan objective of providing appropriate business solutions. It realizes the importance of functional \nknowledge and its impact in developing business solutions. We constantly strive to be a leading \ntechnology firm with profound business and functional knowledge.  The key to the company's \nsuccess is the maintenance of a close working relationship with the clients through ensuring the \nbest possible solutions to their needs; to establish and maintain a thorough knowledge and \nunderstanding of client's objective and help them maximize the benefits.  \n \nWe want to establish ourselves as the best choice in Computing and Information Technology \nServices, Consultancy and Development by offe

2it [01:01, 35.54s/it]

['There is no official method to fast -track the incorporation of a company. However, for foreign -\nowned companies registering with BIDA as industrial ventures, it ma y be noted  that \nBangladesh is in the process of passing legislation to set up a one -stop service centre at BIDA, \nunder which BIDA would assist with the incorporation of foreign -owned companies by \nensuring the completion of registration with RJSC within 48 hours of filing. \n5. What are the main registration  \n \n                                                                               requirements for companies in \nyour \n \nPrivate companies limited by shares (‘private limite d companies’) and public companies \nlimited by s hares (‘public limited companies’) are the most common types of compa nies \nformed in Bangladesh. Section 2(q) of CA 1994 defines a priv ate company as one which \nby its articles restricts the right to transfer its shares,  if any, prohibits any invitation to the \npublic to subsc

Error running evaluator <DynamicRunEvaluator answer_correctness> on run 01a0ba34-974e-7251-b46e-2b501ddc1e4b: RateLimitError("Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kr8f961rf2j93ffaxx5qsc5b` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7640, Requested 1441. Please try again in 8.1075s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}")
Traceback (most recent call last):
  File "c:\Users\Tanvir\AppData\Local\Programs\Python\Python313\Lib\site-packages\langsmith\evaluation\_runner.py", line 1672, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
        run=run,
        example=example,
        evaluator_run_id=evaluator_run_id,
    )
  File "c:\Users\Tanvir\AppData\Local\Programs\Python\Python313\Lib\site-packages\langsmith\evaluation\evaluator.py", line

LangSmith evaluation started.
<ExperimentResults document-rag-v1-dd47994c>
